# Data Ingestion Part 1:
## Day-Ahead Electricity Prices

### Capstone Project for the 2025 "Data Practictioner" Bootcamp" at neuefische
Author: Paul Ringler

This notebook documents the ETL pipeline for day-ahead electricity prices in Austria (2015-2025)

Data is used with permission from EXAA.at
Further details on the data follows in `\capstone\notebooks\08_consolidated_dataset_EDA_and_FeatureEngineering.ipynb`

Objective: Integrate 11 CSV files with 15-minute resolution data into one file. Aggegrate them to daily, weekly and monthly resolution data because all other data is either daily or monthly and EDA and feature engineering will be done only on weekly and monthly data.

In [1]:
#  Setup and Imports
import pandas as pd
import numpy as np
import glob
from pathlib import Path
from datetime import datetime
import warnings


The following code defines `setup_project_paths()` which gives file locations to the ETL pipeline, most critically the following:

    PROJECT_ROOT = Root path of the project folder
    RAW_DATA_PATH = Filepath of the raw data files
    PROCESSED_DATA_PATH = Filepath of the processed data files

In [2]:
# ==============a===============================================================
# NOTEBOOK CONFIGURATION (change for each notebook)
# =============================================================================
NOTEBOOK_NAME = "01: Electricity Data Ingestion"
DATA_SUBFOLDER = "day_ahead_prices"
OBJECTIVE = "Process day-ahead electricity prices (2015-2025)\nOutput: Daily, weekly, and monthly aggregated data"

# =============================================================================
# MODULAR FUNCTIONS (extractable to data_ingestion.py)
# =============================================================================

def setup_pandas_options():
    """Configure pandas display options for better data inspection."""
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_rows', 100)  
    pd.set_option('display.width', None)
    warnings.filterwarnings('ignore')

def setup_project_paths(data_subfolder):
    """
    Set up project directory paths for any data source.
    
    Args:
        data_subfolder (str): Name of subfolder in data/raw/ 
    
    Returns:
        tuple: (PROJECT_ROOT, RAW_DATA_PATH, PROCESSED_DATA_PATH)
    """
    PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
    RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / data_subfolder
    PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed'
    
    # Ensure directories exist
    PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
    
    return PROJECT_ROOT, RAW_DATA_PATH, PROCESSED_DATA_PATH

def print_notebook_header(notebook_name, objective, raw_path, processed_path):
    """Print standardized notebook start information."""
    print("="*60)
    print(f"NOTEBOOK: {notebook_name}")
    print("="*60)
    print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Raw Data Path: {raw_path}")
    print(f"Processed Data Path: {processed_path}")
    print(f"Objective: {objective}")
    print("="*60)

# =============================================================================
# NOTEBOOK EXECUTION
# =============================================================================

# Setup
setup_pandas_options()
PROJECT_ROOT, RAW_DATA_PATH, PROCESSED_DATA_PATH = setup_project_paths(DATA_SUBFOLDER)

# Print info
print_notebook_header(NOTEBOOK_NAME, OBJECTIVE, RAW_DATA_PATH, PROCESSED_DATA_PATH)

NOTEBOOK: 01: Electricity Data Ingestion
Start Time: 2025-09-29 12:47:25
Raw Data Path: c:\Users\paulr\Desktop\Capstone\data\raw\day_ahead_prices
Processed Data Path: c:\Users\paulr\Desktop\Capstone\data\processed
Objective: Process day-ahead electricity prices (2015-2025)
Output: Daily, weekly, and monthly aggregated data


The following code defines `discover_data_files()` which discovers and saves the names and location of the data files to be ingested.
This function uses the file paths given by `setup_project_paths()`.

In [3]:
# =============================================================================
# FILE DISCOVERY CONFIGURATION (change for each notebook)
# =============================================================================
FILE_PATTERN = "Day Ahead Preise_M15_*_English.csv"  # Change per notebook
EXTRACT_YEAR = True  # Set False for single files
YEAR_EXTRACTION_METHOD = "split_pattern"  # Options: "split_pattern", "regex", None

# =============================================================================
# MODULAR FUNCTIONS (extractable to data_ingestion.py)
# =============================================================================

def discover_data_files(data_path, file_pattern="*.csv", extract_year=True, extraction_method="split_pattern"):
    """
    Generic file discovery for any data source.
    
    Args:
        data_path (Path): Path to raw data directory
        file_pattern (str): Glob pattern for file matching
        extract_year (bool): Whether to extract year from filename
        extraction_method (str): How to extract year ("split_pattern", "regex", None)
    
    Returns:
        list: List of dictionaries with file info
    """
    # Find all matching files
    pattern_path = str(data_path / file_pattern)
    found_files = glob.glob(pattern_path)
    
    file_info = []
    for file_path in found_files:
        filename = Path(file_path).name
        
        info = {
            'filename': filename,
            'path': file_path
        }
        
        # Extract year if requested
        if extract_year and extraction_method == "split_pattern":
            try:
                # For electricity files: between M15_ and _English
                year = filename.split('M15_')[1].split('_English')[0]
                info['year'] = int(year)
            except (IndexError, ValueError):
                info['year'] = None
        
        file_info.append(info)
    
    # Sort by year if years were extracted, otherwise by filename
    if extract_year and any(info.get('year') for info in file_info):
        return sorted(file_info, key=lambda x: x.get('year', 0))
    else:
        return sorted(file_info, key=lambda x: x['filename'])

def display_file_discovery_results(file_info, has_years=True):
    """Display results of file discovery in standardized format."""
    print(f"Found {len(file_info)} data files:")
    print("-" * 50)
    
    for info in file_info:
        if has_years and 'year' in info and info['year']:
            print(f"  {info['year']}: {info['filename']}")
        else:
            print(f"  {info['filename']}")
    
    if file_info:
        if has_years and any(info.get('year') for info in file_info):
            years = [info['year'] for info in file_info if info.get('year')]
            if years:
                year_range = f"{min(years)} to {max(years)}"
                print(f"\nYear Range: {year_range}")
        
        print(f"Ready to process {len(file_info)} files.")
    else:
        print("No files found! Check the data path and file pattern.")

# =============================================================================
# NOTEBOOK EXECUTION
# =============================================================================

# Discover and display files
file_info = discover_data_files(RAW_DATA_PATH, FILE_PATTERN, EXTRACT_YEAR, YEAR_EXTRACTION_METHOD)
display_file_discovery_results(file_info, has_years=EXTRACT_YEAR)

Found 11 data files:
--------------------------------------------------
  2015: Day Ahead Preise_M15_2015_English.csv
  2016: Day Ahead Preise_M15_2016_English.csv
  2017: Day Ahead Preise_M15_2017_English.csv
  2018: Day Ahead Preise_M15_2018_English.csv
  2019: Day Ahead Preise_M15_2019_English.csv
  2020: Day Ahead Preise_M15_2020_English.csv
  2021: Day Ahead Preise_M15_2021_English.csv
  2022: Day Ahead Preise_M15_2022_English.csv
  2023: Day Ahead Preise_M15_2023_English.csv
  2024: Day Ahead Preise_M15_2024_English.csv
  2025: Day Ahead Preise_M15_2025_English.csv

Year Range: 2015 to 2025
Ready to process 11 files.


The following code sets up `load_sample_file()`, `analyze_data_structure()` and `display_sample_data(df)`, to display and explore the raw data_file for 2015.

Usage:
'df, sample_file=load_sample_file(file_info)'
'analyze_data_structure(df, sample_file['filename'])'
display_sample_data(df)

In [4]:
# =============================================================================
# MODULAR FUNCTIONS (extractable to data_ingestion.py)
# =============================================================================

def load_sample_file(file_info, index=0, **pd_kwargs):
    """
    Load a sample file with automatic format detection.
    
    Args:
        file_info (list): List of file dictionaries from discover_data_files()
        index (int): Which file to load (default: first file)
        **pd_kwargs: Additional pandas arguments (nrows, encoding, etc.)
    
    Returns:
        tuple: (DataFrame, file_info_dict)
    """
    if not file_info:
        raise ValueError("No files found in file_info list")
    
    sample_file = file_info[index]
    file_path = sample_file['path']
    
    # Auto-detect file format and load
    if file_path.lower().endswith('.xlsx'):
        df = pd.read_excel(file_path, **pd_kwargs)
    elif file_path.lower().endswith('.csv'):
        df = pd.read_csv(file_path, **pd_kwargs)
    else:
        # Try CSV as fallback
        df = pd.read_csv(file_path, **pd_kwargs)
    
    return df, sample_file

def analyze_data_structure(df, sample_file_name):
    """
    Perform universal data structure analysis.
    
    Args:
        df (DataFrame): Data to analyze
        sample_file_name (str): Name of file being analyzed
    
    Returns:
        dict: Analysis results for potential further processing
    """
    print(f"Analyzing sample file: {sample_file_name}")
    print("=" * 60)
    
    # Basic structure
    print("BASIC STRUCTURE:")
    print(f"Shape: {df.shape} (rows x columns)")
    print()
    
    # Columns and data types
    print("COLUMNS AND DATA TYPES:")
    for i, (col, dtype) in enumerate(df.dtypes.items()):
        print(f"  {i+1}. {col}: {dtype}")
    print()
    
    # Missing values analysis
    print("MISSING VALUES PER COLUMN:")
    missing_counts = df.isnull().sum()
    total_missing = 0
    
    for col, count in missing_counts.items():
        if count > 0:
            percentage = count/len(df)*100
            print(f"  {col}: {count} missing ({percentage:.1f}%)")
            total_missing += count
    
    if total_missing == 0:
        print("  No missing values found")
    print()
    
    # Return analysis results for further processing
    return {
        'shape': df.shape,
        'columns': list(df.columns),
        'dtypes': dict(df.dtypes),
        'missing_counts': dict(missing_counts),
        'total_missing': total_missing
    }

def display_sample_data(df, n_rows=5):
    """
    Display sample data (head and tail).
    
    Args:
        df (DataFrame): Data to display
        n_rows (int): Number of rows to show from head/tail
    """
    print(f"FIRST {n_rows} ROWS:")
    print(df.head(n_rows))
    print()
    
    print(f"LAST {n_rows} ROWS:")
    print(df.tail(n_rows))
    print()

In [5]:
df, sample_file=load_sample_file(file_info)
analyze_data_structure(df, sample_file['filename'])


Analyzing sample file: Day Ahead Preise_M15_2015_English.csv
BASIC STRUCTURE:
Shape: (35040, 5) (rows x columns)

COLUMNS AND DATA TYPES:
  1. Time from [CET/CEST]: object
  2. Time to [CET/CEST]: object
  3. Price EXAA 10:15 Auction [EUR/MWh]: float64
  4. Price MC Auction [EUR/MWh]: float64
  5. MC Reference price [EUR/MWh]: float64

MISSING VALUES PER COLUMN:
  Price EXAA 10:15 Auction [EUR/MWh]: 204 missing (0.6%)
  Price MC Auction [EUR/MWh]: 35040 missing (100.0%)
  MC Reference price [EUR/MWh]: 35040 missing (100.0%)



{'shape': (35040, 5),
 'columns': ['Time from [CET/CEST]',
  'Time to [CET/CEST]',
  'Price EXAA 10:15 Auction [EUR/MWh]',
  'Price MC Auction [EUR/MWh]',
  'MC Reference price [EUR/MWh]'],
 'dtypes': {'Time from [CET/CEST]': dtype('O'),
  'Time to [CET/CEST]': dtype('O'),
  'Price EXAA 10:15 Auction [EUR/MWh]': dtype('float64'),
  'Price MC Auction [EUR/MWh]': dtype('float64'),
  'MC Reference price [EUR/MWh]': dtype('float64')},
 'missing_counts': {'Time from [CET/CEST]': 0,
  'Time to [CET/CEST]': 0,
  'Price EXAA 10:15 Auction [EUR/MWh]': 204,
  'Price MC Auction [EUR/MWh]': 35040,
  'MC Reference price [EUR/MWh]': 35040},
 'total_missing': 70284}

In [6]:
display_sample_data(df)

FIRST 5 ROWS:
  Time from [CET/CEST]   Time to [CET/CEST]  \
0  2015-01-01 00:00:00  2015-01-01 00:15:00   
1  2015-01-01 00:15:00  2015-01-01 00:30:00   
2  2015-01-01 00:30:00  2015-01-01 00:45:00   
3  2015-01-01 00:45:00  2015-01-01 01:00:00   
4  2015-01-01 01:00:00  2015-01-01 01:15:00   

   Price EXAA 10:15 Auction [EUR/MWh]  Price MC Auction [EUR/MWh]  \
0                               85.72                         NaN   
1                               67.50                         NaN   
2                               34.64                         NaN   
3                               20.00                         NaN   
4                               88.20                         NaN   

   MC Reference price [EUR/MWh]  
0                           NaN  
1                           NaN  
2                           NaN  
3                           NaN  
4                           NaN  

LAST 5 ROWS:
      Time from [CET/CEST]   Time to [CET/CEST]  \
35035  2015-12-31 2

In [7]:
# =============================================================================
# DOMAIN-SPECIFIC ANALYSIS (Electricity Data)
# =============================================================================

print("DOMAIN-SPECIFIC ANALYSIS:")
print("-" * 30)

# Analyze datetime columns (electricity-specific)
print("SAMPLE DATETIME VALUES:")
datetime_cols = [col for col in df.columns if 'time' in col.lower()]
if len(datetime_cols) >= 2:
    print("Time from examples:", df.iloc[:3, 0].tolist())
    print("Time to examples:", df.iloc[:3, 1].tolist())

# Analyze price columns (electricity-specific)
print("\nPRICE COLUMNS ANALYSIS:")
price_cols = [col for col in df.columns if 'price' in col.lower() or 'EUR' in col]
for col in price_cols:
    non_null_count = df[col].notna().sum()
    print(f"  {col}: {non_null_count}/{len(df)} non-null values")
    
    if non_null_count > 0:
        # Show sample of actual values (first 5 non-null)
        sample_values = df[col].dropna().head(5).tolist()
        print(f"    Sample values: {sample_values}")

print("\n" + "="*60)

DOMAIN-SPECIFIC ANALYSIS:
------------------------------
SAMPLE DATETIME VALUES:
Time from examples: ['2015-01-01 00:00:00', '2015-01-01 00:15:00', '2015-01-01 00:30:00']
Time to examples: ['2015-01-01 00:15:00', '2015-01-01 00:30:00', '2015-01-01 00:45:00']

PRICE COLUMNS ANALYSIS:
  Price EXAA 10:15 Auction [EUR/MWh]: 34836/35040 non-null values
    Sample values: [85.72, 67.5, 34.64, 20.0, 88.2]
  Price MC Auction [EUR/MWh]: 0/35040 non-null values
  MC Reference price [EUR/MWh]: 0/35040 non-null values



There are missing values in the dataset. This will be adressed in subsequent cleaning steps.

#### Structural Break in Data: Availability of EXAA and MC Auction Prices
The Day-Ahead Price raw datasets contains two columns supplying 15-min interval prices:
* `Price EXAA 10:15 Auction [EUR/MWh]`
* `Price MC Auction [EUR/MWh]`

`Price EXAA 10:15 Auction [EUR/MWh]` is available from 2015 to 2022, while `Price MC Auction [EUR/MWh]` is available from 2020 onwards. 
Research on the Austrian Power Grid Website (https://markt.apg.at/transparenz/uebertragung/day-ahead-preise/) shows the following important facts:

1. both prices remain available for visual analysis of historical data from 2023 onwards
2. The 15min EXAA price is rather volatile, showing "micro seasonal" volatility within one hour.
3. The 15min MC Auction price is less volatile, showing no such "micro seasons"
4. Both prices follow similar trend patterns.

While the reason for the structural break in the historical is not immediately apparent, it seems to be due rather to internal policy changes regarding the publication of historical data than external factors like regulation changes.

Conclusion: Since this project focuses on the analysis of weekly and monthly averages and prediction of future monthly prices, this break will be dealt with by using available EXAA data from 2015 to 2020, creating a weighted average for both prices from 2020 to 2022 and then using MC price data from 2023 onwards.

The following cell identifies that structural break using `analyze_price_columns_across_years()` and `display_price_structure_summary`.

In [8]:
import openpyxl 
# =============================================================================
# Structure Analysis Across All Years (2015-2025)
# =============================================================================

# MODULAR FUNCTIONS (extractable to data_ingestion.py)
# =============================================================================

def analyze_price_columns_across_years(file_info, sample_size=1000):
    """
    Analyze price column availability across all years.
    
    Args:
        file_info (list): List of file dictionaries
        sample_size (int): Number of rows to sample per file (for speed)
    
    Returns:
        DataFrame: Structured analysis results
    """
    analysis_results = []
    
    for file_dict in file_info:
        year = file_dict.get('year', 'Unknown')
        filename = file_dict['filename']
        file_path = file_dict['path']
        
        print(f"Analyzing {year}: {filename}")
        
        try:
            # Direct pandas loading - much cleaner!
            df = pd.read_csv(file_path, nrows=sample_size)
            
            # Find price columns
            price_columns = [col for col in df.columns if 'price' in col.lower() or 'EUR' in col]
            
            year_data = {
                'year': year,
                'filename': filename,
                'total_rows_sampled': len(df),
                'total_columns': len(df.columns)
            }
            
            # Analyze each price column
            for col in price_columns:
                # Clean column name for DataFrame column naming
                clean_col_name = col.replace(' ', '_').replace('[', '').replace(']', '').replace('/', '_')
                
                non_null_count = df[col].notna().sum()
                percentage = (non_null_count / len(df)) * 100 if len(df) > 0 else 0
                
                year_data[f'{clean_col_name}_available'] = non_null_count > 0
                year_data[f'{clean_col_name}_non_null_count'] = non_null_count
                year_data[f'{clean_col_name}_percentage'] = round(percentage, 1)
                
                # Sample of actual values if available
                if non_null_count > 0:
                    sample_vals = df[col].dropna().head(3).tolist()
                    year_data[f'{clean_col_name}_sample_values'] = str(sample_vals)
                else:
                    year_data[f'{clean_col_name}_sample_values'] = "No data"
            
            analysis_results.append(year_data)
            
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            # Add error record
            analysis_results.append({
                'year': year,
                'filename': filename,
                'error': str(e)
            })
    
    return pd.DataFrame(analysis_results)

def display_price_structure_summary(structure_df):
    """Display summary of price column availability across years."""
    print("\n" + "="*80)
    print("PRICE STRUCTURE SUMMARY ACROSS ALL YEARS")
    print("="*80)
    
    # Show availability by year
    for _, row in structure_df.iterrows():
        year = row['year']
        print(f"\n{year}:")
        
        # Find available price columns
        available_cols = []
        for col in row.index:
            if col.endswith('_available') and row[col] == True:
                price_col_name = col.replace('_available', '')
                non_null_col = f'{price_col_name}_non_null_count'
                percentage_col = f'{price_col_name}_percentage'
                
                if non_null_col in row.index:
                    count = row[non_null_col]
                    percentage = row[percentage_col]
                    available_cols.append(f"{price_col_name}: {count} values ({percentage}%)")
        
        if available_cols:
            print("\n".join(available_cols))
        else:
            print("No price data available")

# =============================================================================
# NOTEBOOK EXECUTION
# =============================================================================

print("Analyzing price column structure across all years...")
print("Sampling 1000 rows per file for faster processing...")
print("-" * 60)

# Perform analysis
structure_df = analyze_price_columns_across_years(file_info, sample_size=1000)

# Display summary
display_price_structure_summary(structure_df)

# Export to Excel for detailed inspection
excel_path = PROCESSED_DATA_PATH / "electricity_price_structure_analysis.xlsx"
structure_df.to_excel(excel_path, index=False)

print(f"\nDetailed analysis exported to: {excel_path}")
print(f"DataFrame shape: {structure_df.shape}")
print("\nFirst few columns of analysis DataFrame:")
print(structure_df[['year', 'filename', 'total_columns']].head())

Analyzing price column structure across all years...
Sampling 1000 rows per file for faster processing...
------------------------------------------------------------
Analyzing 2015: Day Ahead Preise_M15_2015_English.csv
Analyzing 2016: Day Ahead Preise_M15_2016_English.csv
Analyzing 2017: Day Ahead Preise_M15_2017_English.csv
Analyzing 2018: Day Ahead Preise_M15_2018_English.csv
Analyzing 2019: Day Ahead Preise_M15_2019_English.csv
Analyzing 2020: Day Ahead Preise_M15_2020_English.csv
Analyzing 2021: Day Ahead Preise_M15_2021_English.csv
Analyzing 2022: Day Ahead Preise_M15_2022_English.csv
Analyzing 2023: Day Ahead Preise_M15_2023_English.csv
Analyzing 2024: Day Ahead Preise_M15_2024_English.csv
Analyzing 2025: Day Ahead Preise_M15_2025_English.csv

PRICE STRUCTURE SUMMARY ACROSS ALL YEARS

2015:
Price_EXAA_10:15_Auction_EUR_MWh: 1000.0 values (100.0%)

2016:
Price_EXAA_10:15_Auction_EUR_MWh: 1000.0 values (100.0%)

2017:
Price_EXAA_10:15_Auction_EUR_MWh: 1000.0 values (100.0%)

2018

#### Further manual preparatory Data Exploration
The next step included around 4 hours of manual preparatory data exploration outside of the python notebook.
This was done to ensure that the modular data ingestion, processing and consolidation functions would be capable of executing all required functions while dealing with highly varied data structures.
For two datasets (the economic data from STATISTIK AUSTRIA and the Gas Price Index from AEA), manual restructuring of the data was performed (removal of .jpeg thumbnails and cropping of the file to make the headers the first row in the file).
Further details on the results of this step can be found in this file `\capstone\notebooks\Data_Exploration_For_Processing_And_Consolidation_25_09_17.txt`

#### ETL PIPELINE for price data

The following steps set up the ETL pipeline for the pricing data.

The overall setup looks like this:

**Phase 1: Discovery**
```
discover_electricity_files()
- Input: Raw data directory path  
- Output: List[{year, filename, path}]
- Next Phase: Loading
```

**Phase 2: Loading**
```
load_all_electricity_files()
- Input: file_info (from Phase 1)
- Uses: standardize_missing_values()
- Output: Dict[year: (DataFrame, metadata)]
- Next Phase: Standardization
```

**Phase 3: Standardization**
```
extract_hybrid_price_columns() 
- Input: loaded_files (from Phase 2)
- Output: Dict[year: standardized_DataFrame]
- Next Phase: Aggregation
```

**Phase 4: Aggregation**
```
create_time_aggregations()
- Input: processed_files (from Phase 3)  
- Output: Dict[year: {daily, weekly, monthly}]
- Next Phase: Transformation
```

**Phase 5: Transformation**
```
transform_to_long_format()
- Input: aggregated_files (from Phase 4)
- Output: Final long-format DataFrame
- Next Phase: Consolidation and Export
```

**Phase 6: Orchestration & Export**
```
consolidate_electricity_data() [MAIN FUNCTION]
- Execute all Phases 1-5 
- Also: setup_pandas_options()
- Output: CSV-Export + DataFrame return
- End of Pipeline 
```

**Function Call Chain:**

```
consolidate_electricity_data()
    |
    +-- setup_pandas_options()
    |
    +-- discover_electricity_files()
    |       |
    |       v
    +-- load_all_electricity_files()
    |       |
    |       +-- standardize_missing_values() [for each file]
    |       |
    |       v
    +-- extract_hybrid_price_columns() [for each year]
    |       |
    |       v
    +-- create_time_aggregations() [for each year]
    |       |
    |       v
    +-- transform_to_long_format() [combine all years]
```

**Data Flow:**

```
Raw CSV Files
    -> discover_electricity_files()
File Info List  
    -> load_all_electricity_files() + standardize_missing_values()
Loaded DataFrames Dict
    -> extract_hybrid_price_columns()
Standardized Price DataFrames
    -> create_time_aggregations()  
Aggregated Data by Year/Level
    -> transform_to_long_format()
Final Analysis-Ready Dataset
```

In [21]:
# =============================================================================
# CELL 2: Missing Values Standardization Function (with Quality Control)
# =============================================================================

def standardize_missing_values(df, additional_missing=None, show_quality_control=True):
    """
    Convert various missing value representations to pandas NaN.
    
    Args:
        df (DataFrame): Input dataframe
        additional_missing (list): Extra missing indicators beyond defaults
        show_quality_control (bool): Whether to display quality control output
    
    Returns:
        DataFrame: Dataframe with standardized missing values
        dict: Report of found missing patterns
    """
    missing_indicators = [
        'N/A', 'n/a', 'NA', 'na',
        '', '-', '--', '---',
        'NULL', 'null', 'Null',
        'NaN', 'nan', '#N/A'
    ]
    
    if additional_missing:
        missing_indicators.extend(additional_missing)
    
    if show_quality_control:
        print("\nMISSING VALUES STANDARDIZATION - QUALITY CONTROL:")
        print("-" * 55)
        print("Missing indicators standardized:")
        print("  ", ', '.join([f"'{ind}'" for ind in missing_indicators]))
        print()
    
    found_patterns = {}
    df_clean = df.copy()
    
    for col in df_clean.columns:
        original_nulls = df_clean[col].isnull().sum()
        df_clean[col] = df_clean[col].replace(missing_indicators, np.nan)
        new_nulls = df_clean[col].isnull().sum()
        converted_count = new_nulls - original_nulls
        
        if converted_count > 0:
            found_patterns[col] = {
                'original_nulls': original_nulls,
                'converted_missing': converted_count,
                'total_nulls': new_nulls
            }
    
    if show_quality_control:
        if found_patterns:
            print("CONVERSION RESULTS BY COLUMN:")
            for col, pattern in found_patterns.items():
                print(f"  {col}:")
                print(f"    Original nulls: {pattern['original_nulls']}")
                print(f"    Converted missing: {pattern['converted_missing']}")
                print(f"    Total nulls: {pattern['total_nulls']}")
        else:
            print("No missing value patterns found for conversion")
        print("-" * 55)
    
    return df_clean, found_patterns

print("Cell 2: Missing Values Standardization Function (with Quality Control) - READY")

Cell 2: Missing Values Standardization Function (with Quality Control) - READY


In [10]:
# =============================================================================
# CELL 6: Load All Electricity Files
# =============================================================================

def load_all_electricity_files(file_info):
    """
    Load all 11 electricity files with automatic error handling.
    
    Args:
        file_info (list): List of file dictionaries from discover_electricity_files
    
    Returns:
        dict: Dictionary with year as key, (dataframe, metadata) as value
    """
    loaded_files = {}
    
    for file_dict in file_info:
        year = file_dict['year']
        file_path = file_dict['path']
        
        print(f"Loading {year}: {file_dict['filename']}")
        
        try:
            # Try UTF-8 first, fall back to cp1252
            try:
                df = pd.read_csv(file_path, encoding='utf-8')
            except UnicodeDecodeError:
                df = pd.read_csv(file_path, encoding='cp1252')
            
            # Clean missing values
            df_clean, missing_patterns = standardize_missing_values(df)
            
            # Convert time columns to datetime
            time_columns = [col for col in df_clean.columns if 'time' in col.lower()]
            for col in time_columns:
                df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
            
            # Create metadata
            metadata = {
                'year': year,
                'filename': file_dict['filename'],
                'rows': len(df_clean),
                'columns': list(df_clean.columns),
                'price_columns': [col for col in df_clean.columns if 'price' in col.lower() or 'EUR' in col],
                'missing_patterns_found': missing_patterns,
                'date_range': (df_clean.iloc[:, 0].min(), df_clean.iloc[:, 0].max()) if len(df_clean) > 0 else (None, None)
            }
            
            loaded_files[year] = (df_clean, metadata)
            print(f"  Success: {len(df_clean)} rows, {len(metadata['price_columns'])} price columns")
            
        except Exception as e:
            print(f"  ERROR loading {year}: {e}")
            loaded_files[year] = (None, {'error': str(e)})
    
    return loaded_files

print("Cell 6: Load All Electricity Files Function - READY")

Cell 6: Load All Electricity Files Function - READY


In [11]:
# =============================================================================
# CELL 7: Extract Hybrid Price Columns
# =============================================================================

def extract_hybrid_price_columns(df, year, metadata):
    """
    Extract correct price columns based on year and rename consistently.
    
    Args:
        df (DataFrame): Raw electricity data
        year (int): Year of the data
        metadata (dict): Metadata about the file
    
    Returns:
        DataFrame: DataFrame with standardized price columns
    """
    if df is None:
        return None
    
    # Create working copy
    df_prices = df.copy()
    
    # Initialize price columns as None
    df_prices['price_exaa_raw'] = np.nan
    df_prices['price_mc_raw'] = np.nan
    
    # Extract EXAA prices (2015-2022)
    if 2015 <= year <= 2022:
        exaa_col = None
        for col in df.columns:
            if 'EXAA' in col and 'EUR' in col:
                exaa_col = col
                break
        
        if exaa_col:
            df_prices['price_exaa_raw'] = pd.to_numeric(df[exaa_col], errors='coerce')
    
    # Extract MC Auction prices (2020-2025)
    if 2020 <= year <= 2025:
        mc_col = None
        for col in df.columns:
            if 'MC Auction' in col and 'EUR' in col:
                mc_col = col
                break
        
        if mc_col:
            df_prices['price_mc_raw'] = pd.to_numeric(df[mc_col], errors='coerce')
    
    # Keep only essential columns: time + prices
    time_col = df.columns[0]  # First column should be "Time from"
    essential_columns = [time_col, 'price_exaa_raw', 'price_mc_raw']
    
    df_final = df_prices[essential_columns].copy()
    df_final.rename(columns={time_col: 'timestamp'}, inplace=True)
    
    return df_final

print("Cell 7: Extract Hybrid Price Columns Function - READY")


Cell 7: Extract Hybrid Price Columns Function - READY


In [12]:
# Test the loading function
print("Testing Cell 6...")
loaded_files = load_all_electricity_files(file_info)
print(f"Loaded {len(loaded_files)} files")

# Show what we got
for year, (df, metadata) in loaded_files.items():
    if df is not None:
        print(f"  {year}: {metadata['rows']} rows, columns: {metadata['price_columns']}")
    else:
        print(f"  {year}: FAILED - {metadata.get('error', 'Unknown error')}")

print("Cell 6B: Loading Test - COMPLETE")

Testing Cell 6...
Loading 2015: Day Ahead Preise_M15_2015_English.csv
  Success: 35040 rows, 3 price columns
Loading 2016: Day Ahead Preise_M15_2016_English.csv
  Success: 35136 rows, 3 price columns
Loading 2017: Day Ahead Preise_M15_2017_English.csv
  Success: 35040 rows, 3 price columns
Loading 2018: Day Ahead Preise_M15_2018_English.csv
  Success: 35040 rows, 3 price columns
Loading 2019: Day Ahead Preise_M15_2019_English.csv
  Success: 35040 rows, 3 price columns
Loading 2020: Day Ahead Preise_M15_2020_English.csv
  Success: 35136 rows, 3 price columns
Loading 2021: Day Ahead Preise_M15_2021_English.csv
  Success: 35040 rows, 3 price columns
Loading 2022: Day Ahead Preise_M15_2022_English.csv
  Success: 35040 rows, 3 price columns
Loading 2023: Day Ahead Preise_M15_2023_English.csv
  Success: 35040 rows, 2 price columns
Loading 2024: Day Ahead Preise_M15_2024_English.csv
  Success: 35136 rows, 2 price columns
Loading 2025: Day Ahead Preise_M15_2025_English.csv
  Success: 23324 row

In [13]:
# =============================================================================
# CELL 7: Extract Hybrid Price Columns
# =============================================================================

def extract_hybrid_price_columns(df, year, metadata):
    """
    Extract correct price columns based on year and rename consistently.
    
    Args:
        df (DataFrame): Raw electricity data
        year (int): Year of the data
        metadata (dict): Metadata about the file
    
    Returns:
        DataFrame: DataFrame with standardized price columns
    """
    if df is None:
        return None
    
    # Create working copy
    df_prices = df.copy()
    
    # Initialize price columns as None
    df_prices['price_exaa_raw'] = np.nan
    df_prices['price_mc_raw'] = np.nan
    
    # Extract EXAA prices (2015-2022)
    if 2015 <= year <= 2022:
        exaa_col = None
        for col in df.columns:
            if 'EXAA' in col and 'EUR' in col:
                exaa_col = col
                break
        
        if exaa_col:
            df_prices['price_exaa_raw'] = pd.to_numeric(df[exaa_col], errors='coerce')
    
    # Extract MC Auction prices (2020-2025)
    if 2020 <= year <= 2025:
        mc_col = None
        for col in df.columns:
            if 'MC Auction' in col and 'EUR' in col:
                mc_col = col
                break
        
        if mc_col:
            df_prices['price_mc_raw'] = pd.to_numeric(df[mc_col], errors='coerce')
    
    # Keep only essential columns: time + prices
    time_col = df.columns[0]  # First column should be "Time from"
    essential_columns = [time_col, 'price_exaa_raw', 'price_mc_raw']
    
    df_final = df_prices[essential_columns].copy()
    df_final.rename(columns={time_col: 'timestamp'}, inplace=True)
    
    return df_final

print("Cell 7: Extract Hybrid Price Columns Function - READY")

Cell 7: Extract Hybrid Price Columns Function - READY


In [14]:
# =============================================================================
# CELL 8: Create Time Aggregations
# =============================================================================

def create_time_aggregations(df):
    """
    Create daily, weekly, and monthly aggregations from 15-minute data.
    
    Args:
        df (DataFrame): DataFrame with timestamp and price columns
    
    Returns:
        dict: Dictionary with 'daily', 'weekly', 'monthly' DataFrames
    """
    if df is None or len(df) == 0:
        return {'daily': None, 'weekly': None, 'monthly': None}
    
    # Ensure timestamp is datetime and set as index
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df_indexed = df.set_index('timestamp')
    
    aggregations = {}
    
    # Daily aggregation
    daily = df_indexed.resample('D').agg({
        'price_exaa_raw': ['mean', 'count'],
        'price_mc_raw': ['mean', 'count']
    })
    daily.columns = ['price_exaa_mean', 'price_exaa_count', 'price_mc_mean', 'price_mc_count']
    daily['date'] = daily.index.date
    daily['aggregation_level'] = 'daily'
    aggregations['daily'] = daily.reset_index(drop=True)
    
    # Weekly aggregation (ISO weeks, Monday start)
    weekly = df_indexed.resample('W-MON').agg({
        'price_exaa_raw': ['mean', 'count'],
        'price_mc_raw': ['mean', 'count']
    })
    weekly.columns = ['price_exaa_mean', 'price_exaa_count', 'price_mc_mean', 'price_mc_count']
    # Use Monday of each week as the date
    weekly['date'] = weekly.index.date
    weekly['aggregation_level'] = 'weekly'
    aggregations['weekly'] = weekly.reset_index(drop=True)
    
    # Monthly aggregation (first of month)
    monthly = df_indexed.resample('MS').agg({
        'price_exaa_raw': ['mean', 'count'],
        'price_mc_raw': ['mean', 'count']
    })
    monthly.columns = ['price_exaa_mean', 'price_exaa_count', 'price_mc_mean', 'price_mc_count']
    monthly['date'] = monthly.index.date
    monthly['aggregation_level'] = 'monthly'
    aggregations['monthly'] = monthly.reset_index(drop=True)
    
    return aggregations

print("Cell 8: Create Time Aggregations Function - READY")

Cell 8: Create Time Aggregations Function - READY


In [15]:
# =============================================================================
# CELL 9: Transform to Long Format
# =============================================================================

def transform_to_long_format(aggregated_data_dict):
    """
    Transform aggregated data to final long format with all required columns.
    
    Args:
        aggregated_data_dict (dict): Dictionary of aggregated DataFrames by year
    
    Returns:
        DataFrame: Final long-format DataFrame
    """
    all_data = []
    
    for year, agg_data in aggregated_data_dict.items():
        if agg_data is None:
            continue
            
        for level in ['daily', 'weekly', 'monthly']:
            df = agg_data[level]
            if df is None or len(df) == 0:
                continue
            
            # Convert date to datetime for calculations
            df['date'] = pd.to_datetime(df['date'])
            
            # Add time components
            df['year'] = df['date'].dt.year
            df['month'] = df['date'].dt.month  
            df['quarter'] = df['date'].dt.quarter
            df['week'] = df['date'].dt.isocalendar().week
            df['month_name'] = df['date'].dt.month_name()
            
            # Round prices to integers
            df['price_exaa_mean'] = df['price_exaa_mean'].round().astype('Int64')
            df['price_mc_auction_mean'] = df['price_mc_mean'].round().astype('Int64')
            
            # Rename count columns
            df['price_count_exaa'] = df['price_exaa_count']
            df['price_count_mc'] = df['price_mc_count']
            
            # Calculate data completeness (% of expected intervals with data)
            expected_intervals = {
                'daily': 96,     # 96 x 15-min intervals per day
                'weekly': 672,   # 7 days x 96 intervals
                'monthly': 2976  # ~31 days x 96 intervals (approximate)
            }
            
            total_count = df['price_count_exaa'].fillna(0) + df['price_count_mc'].fillna(0)
            df['data_completeness'] = (total_count / expected_intervals[level] * 100).round(1)
            
            # Convert date back to string in ISO format
            df['date'] = df['date'].dt.strftime('%Y-%m-%d')
            
            # Select final columns
            final_columns = [
                'date', 'year', 'month', 'quarter', 'week', 'aggregation_level',
                'price_exaa_mean', 'price_mc_auction_mean', 
                'price_count_exaa', 'price_count_mc', 
                'data_completeness', 'month_name'
            ]
            
            df_final = df[final_columns].copy()
            all_data.append(df_final)
    
    # Concatenate all data
    if all_data:
        result = pd.concat(all_data, ignore_index=True)
        # Sort by date and aggregation level
        result = result.sort_values(['date', 'aggregation_level']).reset_index(drop=True)
        
        # Clean up: ensure only final columns remain
        final_columns = [
            'date', 'year', 'month', 'quarter', 'week', 'aggregation_level',
            'price_exaa_mean', 'price_mc_auction_mean', 
            'price_count_exaa', 'price_count_mc', 
            'data_completeness', 'month_name'
        ]
        
        # Only keep columns that exist and are in our final list
        existing_final_columns = [col for col in final_columns if col in result.columns]
        result_clean = result[existing_final_columns].copy()
        
        return result_clean
    else:
        return pd.DataFrame()

print("Cell 9: Transform to Long Format Function - READY")

Cell 9: Transform to Long Format Function - READY


In [16]:
# =============================================================================
# CELL 10: Master Orchestration
# =============================================================================

def consolidate_electricity_data_complete(file_info, output_dir):
    """
    Master function to orchestrate complete electricity data consolidation.
    
    Args:
        file_info (list): List of file dictionaries
        output_dir (Path): Directory for outputs
    
    Returns:
        DataFrame: Final consolidated dataset
    """
    print("="*60)
    print("ELECTRICITY DATA CONSOLIDATION - MASTER PROCESS")
    print("="*60)
    
    # Step 1: Load all files
    print("\nSTEP 1: Loading all electricity files...")
    loaded_files = load_all_electricity_files(file_info)
    
    # Step 2: Extract hybrid price columns
    print("\nSTEP 2: Extracting hybrid price columns...")
    processed_files = {}
    
    for year, (df, metadata) in loaded_files.items():
        if df is not None:
            processed_df = extract_hybrid_price_columns(df, year, metadata)
            if processed_df is not None:
                print(f"  {year}: Processed {len(processed_df)} rows")
                processed_files[year] = processed_df
            else:
                print(f"  {year}: FAILED - No data after processing")
        else:
            print(f"  {year}: SKIPPED - Load error")
    
    # Step 3: Create aggregations
    print("\nSTEP 3: Creating time aggregations...")
    aggregated_files = {}
    
    for year, df in processed_files.items():
        aggregations = create_time_aggregations(df)
        aggregated_files[year] = aggregations
        
        # Report aggregation success
        for level, agg_df in aggregations.items():
            if agg_df is not None:
                print(f"  {year} {level}: {len(agg_df)} periods")
    
    # Step 4: Transform to long format
    print("\nSTEP 4: Transforming to long format...")
    final_df = transform_to_long_format(aggregated_files)
    
    if len(final_df) > 0:
        print(f"  SUCCESS: {len(final_df)} total rows in final dataset")
        print(f"  Date range: {final_df['date'].min()} to {final_df['date'].max()}")
        print(f"  Aggregation levels: {final_df['aggregation_level'].value_counts().to_dict()}")
        print(f"  Columns: {final_df.columns.tolist()}")
        print(f"  Shape: {final_df.shape}")
    else:
        print("  ERROR: No data in final dataset")
        return pd.DataFrame()
    
    # SKIP VALIDATION SAMPLE FOR TESTING
    print("\nSKIPPING STEP 5: Validation sample creation (for testing)")
    
    # Step 6: Save final dataset
    final_path = output_dir / "capstone_df.csv"
    final_df.to_csv(final_path, index=False)
    
    print(f"\nFINAL OUTPUT:")
    print(f"  Main dataset: {final_path}")
    print(f"  Total rows: {len(final_df)}")
    print(f"  Total columns: {len(final_df.columns)}")
    
    return final_df

print("Cell 11: Master Orchestration Function - READY")
print("\n" + "="*60)
print("ALL FUNCTIONS READY FOR TESTING")
print("Use: final_df = consolidate_electricity_data_complete(file_info, PROCESSED_DATA_PATH)")
print("="*60)
print("\n" + "="*60)


Cell 11: Master Orchestration Function - READY

ALL FUNCTIONS READY FOR TESTING
Use: final_df = consolidate_electricity_data_complete(file_info, PROCESSED_DATA_PATH)



In [17]:
final_df = consolidate_electricity_data_complete(file_info, PROCESSED_DATA_PATH)

ELECTRICITY DATA CONSOLIDATION - MASTER PROCESS

STEP 1: Loading all electricity files...
Loading 2015: Day Ahead Preise_M15_2015_English.csv
  Success: 35040 rows, 3 price columns
Loading 2016: Day Ahead Preise_M15_2016_English.csv
  Success: 35136 rows, 3 price columns
Loading 2017: Day Ahead Preise_M15_2017_English.csv
  Success: 35040 rows, 3 price columns
Loading 2018: Day Ahead Preise_M15_2018_English.csv
  Success: 35040 rows, 3 price columns
Loading 2019: Day Ahead Preise_M15_2019_English.csv
  Success: 35040 rows, 3 price columns
Loading 2020: Day Ahead Preise_M15_2020_English.csv
  Success: 35136 rows, 3 price columns
Loading 2021: Day Ahead Preise_M15_2021_English.csv
  Success: 35040 rows, 3 price columns
Loading 2022: Day Ahead Preise_M15_2022_English.csv
  Success: 35040 rows, 3 price columns
Loading 2023: Day Ahead Preise_M15_2023_English.csv
  Success: 35040 rows, 2 price columns
Loading 2024: Day Ahead Preise_M15_2024_English.csv
  Success: 35136 rows, 2 price columns


In [18]:
print("\nFinal DataFrame shape:", final_df.shape)


Final DataFrame shape: (4590, 12)


In [19]:
print(final_df.head())

         date  year  month  quarter  week aggregation_level  price_exaa_mean  \
0  2015-01-01  2015      1        1     1             daily               44   
1  2015-01-01  2015      1        1     1           monthly               30   
2  2015-01-02  2015      1        1     1             daily               31   
3  2015-01-03  2015      1        1     1             daily               19   
4  2015-01-04  2015      1        1     1             daily               14   

   price_mc_auction_mean  price_count_exaa  price_count_mc  data_completeness  \
0                   <NA>                96               0              100.0   
1                   <NA>              2784               0               93.5   
2                   <NA>                96               0              100.0   
3                   <NA>                96               0              100.0   
4                   <NA>                96               0              100.0   

  month_name  
0    January  
1 

In [20]:
print(final_df['aggregation_level'].value_counts())

aggregation_level
daily      3896
weekly      566
monthly     128
Name: count, dtype: int64
